# Notebook 03 — Phase 1B Model Comparison Skeleton

This is the Phase 1B model comparison skeleton for LSTM, TCN, and standard DLinear baselines under no-trade-band binary labels.

Default mode is smoke-only: no training, no checkpoint writing, and no result artifact writing. Full Colab execution is deferred to a later explicit step.

In [ ]:
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd

FULL_RUN = False
RUN_TRAINING = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_ROOT = Path("/content/drive/MyDrive/stockdata")
RAW_DATA_DIR = DATA_ROOT / "Dow_30_1min"
OUTPUT_DIR = DATA_ROOT / "phase1b_notebook03_model_comparison"


In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from ml_utils.config import DataConfig
from ml_utils.dataset import WindowedClassificationDataset
from ml_utils.dataset import fit_scaler_on_train
from ml_utils.dataset import make_no_trade_band_labels
from ml_utils.dataset import make_time_splits
from ml_utils.dataset import transform_split
from ml_utils.dataset import trim_labels_at_split_boundary
from ml_utils.metrics import always_predict_baseline_metrics
from ml_utils.metrics import compute_classification_metrics
from ml_utils.metrics import dummy_baseline_metrics
from ml_utils.models.lstm_classifier import LSTMClassifier
from ml_utils.models.tcn_classifier import TCNClassifier
from ml_utils.models.dlinear_classifier import DLinearClassifier


In [ ]:
TICKERS = ["CSCO", "JPM", "KO", "MSFT", "WMT"]
SEEDS = [42, 43, 44]

WINDOW_SIZE = 12
LABEL_HORIZON_K = 12
THRESHOLD_BPS = 5

NUM_CLASSES = 2
INPUT_SIZE = 12

BATCH_SIZE = 512
LEARNING_RATE = 1e-4
MAX_EPOCHS = 20
EARLY_STOP_PATIENCE = 5

TIMESTAMP_COL = "timestamp"
TICKER_COL = "ticker"
LABEL_COL = "label"
PRICE_COL = "close"
SCALER_TYPE = "standard"

RAW_COLUMNS = ["Date", "Time", "Open", "High", "Low", "Close", "Volume"]

FEATURE_COLS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "macd",
    "macd_signal",
    "macd_hist",
    "rsi_14",
    "bb_pctb",
    "rolling_std_20",
    "obv_roc",
]

MARKET_OPEN = "09:30"
MARKET_CLOSE = "16:00"


## Historical LSTM Context

P1B.10 Candidate A single-seed result:

- `model_macro_f1_mean` ≈ 0.5203
- `dummy_stratified_macro_f1_mean` ≈ 0.4990
- delta ≈ +0.0213

P1B.11b strict multi-seed result:

- seed 42 delta ≈ +0.0213
- seed 43 delta ≈ +0.0038
- seed 44 delta ≈ -0.0089
- overall mean delta ≈ +0.0054
- std ≈ 0.0152

Interpretation: LSTM is weak, seed-sensitive, and not robustly better than dummy. These values are historical context, not Notebook 03 output.

In [ ]:
MODEL_REGISTRY = [
    {
        "model_name": "lstm",
        "import_path": "ml_utils.models.lstm_classifier.LSTMClassifier",
        "constructor": LSTMClassifier,
        "config": {
            "input_size": INPUT_SIZE,
            "hidden_size": 64,
            "num_layers": 2,
            "dropout": 0.2,
            "num_classes": NUM_CLASSES,
        },
    },
    {
        "model_name": "tcn",
        "import_path": "ml_utils.models.tcn_classifier.TCNClassifier",
        "constructor": TCNClassifier,
        "config": {
            "input_size": INPUT_SIZE,
            "num_channels": [32, 32],
            "kernel_size": 3,
            "dropout": 0.1,
            "num_classes": NUM_CLASSES,
            "causal": True,
        },
    },
    {
        "model_name": "dlinear",
        "import_path": "ml_utils.models.dlinear_classifier.DLinearClassifier",
        "constructor": DLinearClassifier,
        "config": dict(
            seq_len=WINDOW_SIZE,
            input_size=INPUT_SIZE,
            num_classes=NUM_CLASSES,
            moving_avg_kernel=5,
            individual=False,
        ),
    },
]


In [ ]:
model_registry_df = pd.DataFrame(
    [
        {
            "model_name": item["model_name"],
            "import_path": item["import_path"],
            "config": item["config"],
        }
        for item in MODEL_REGISTRY
    ]
)
model_registry_df


In [ ]:
NOTEBOOK03_RESULT_COLUMNS = [
    "model_name",
    "ticker",
    "seed",
    "window_size",
    "label_horizon_k",
    "threshold_bps",
    "split",
    "n_train_windows",
    "n_val_windows",
    "n_test_windows",
    "label_retained_pct",
    "model_macro_f1",
    "model_balanced_accuracy",
    "dummy_stratified_macro_f1_mean",
    "dummy_stratified_macro_f1_std",
    "dummy_prior_macro_f1",
    "always_up_macro_f1",
    "always_down_macro_f1",
    "delta_macro_f1_vs_dummy",
    "confusion_matrix",
]


In [ ]:
results_schema_df = pd.DataFrame(columns=NOTEBOOK03_RESULT_COLUMNS)
results_schema_df


In [ ]:
ARTIFACT_NAMES = {
    "per_ticker_results": "notebook03_per_ticker_results.csv",
    "summary_by_model": "notebook03_summary_by_model.csv",
    "summary_by_seed": "notebook03_summary_by_seed.csv",
    "run_manifest": "notebook03_run_manifest.json",
}

RUN_MANIFEST_TEMPLATE = {
    "notebook": "03_model_comparison.ipynb",
    "phase": "P1B.15",
    "git_commit_hash": None,
    "full_run": FULL_RUN,
    "run_training": RUN_TRAINING,
    "models": ["lstm", "tcn", "dlinear"],
    "tickers": TICKERS,
    "seeds": SEEDS,
    "window_size": WINDOW_SIZE,
    "label_horizon_k": LABEL_HORIZON_K,
    "threshold_bps": THRESHOLD_BPS,
}


In [ ]:
artifact_manifest_df = pd.DataFrame(
    [{"artifact_key": key, "filename": value} for key, value in ARTIFACT_NAMES.items()]
)
artifact_manifest_df


In [ ]:
def raw_path_for_ticker(ticker: str) -> Path:
    return RAW_DATA_DIR / f"{ticker}.txt"


def load_one_minute_data(ticker: str, max_rows: int | None = None) -> pd.DataFrame:
    input_path = raw_path_for_ticker(ticker)
    if not input_path.exists():
        raise FileNotFoundError(f"Missing raw 1-minute txt file: {input_path}")

    nrows = max_rows if max_rows is None or max_rows > 0 else 0
    frame = pd.read_csv(input_path, header=None, names=RAW_COLUMNS, nrows=nrows)
    frame = frame[frame["Date"].astype(str).str.lower() != "date"].reset_index(drop=True)
    frame[TIMESTAMP_COL] = pd.to_datetime(
        frame["Date"].astype(str) + " " + frame["Time"].astype(str),
        format="%m/%d/%Y %H:%M",
    )
    frame = frame.drop(columns=["Date", "Time"])
    frame = frame.rename(
        columns={
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )
    numeric_cols = ["open", "high", "low", "close", "volume"]
    frame[numeric_cols] = frame[numeric_cols].apply(pd.to_numeric, errors="raise")
    return frame[[TIMESTAMP_COL, *numeric_cols]].reset_index(drop=True)


In [ ]:
def filter_regular_hours(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy(deep=True)
    if TIMESTAMP_COL not in result.columns:
        raise ValueError(f"Missing required timestamp column: {TIMESTAMP_COL}")
    result[TIMESTAMP_COL] = pd.to_datetime(result[TIMESTAMP_COL])
    market_open = pd.to_datetime(MARKET_OPEN).time()
    market_close = pd.to_datetime(MARKET_CLOSE).time()
    times = result[TIMESTAMP_COL].dt.time
    regular_hours = result[(times >= market_open) & (times <= market_close)]
    return regular_hours.reset_index(drop=True)


def resample_to_five_minutes(df: pd.DataFrame) -> pd.DataFrame:
    required_cols = [TIMESTAMP_COL, "open", "high", "low", "close", "volume"]
    missing_cols = [column for column in required_cols if column not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns for 5-minute resampling: {missing_cols}")

    resampled = (
        df.copy(deep=True)
        .set_index(TIMESTAMP_COL)
        .resample("5min")
        .agg(
            {
                "open": "first",
                "high": "max",
                "low": "min",
                "close": "last",
                "volume": "sum",
            }
        )
        .dropna(subset=["open", "high", "low", "close", "volume"])
        .reset_index()
    )
    return filter_regular_hours(resampled)


In [ ]:
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy(deep=True)

    ema12 = result["close"].ewm(span=12, adjust=False).mean()
    ema26 = result["close"].ewm(span=26, adjust=False).mean()
    result["macd"] = ema12 - ema26
    result["macd_signal"] = result["macd"].ewm(span=9, adjust=False).mean()
    result["macd_hist"] = result["macd"] - result["macd_signal"]

    close_delta = result["close"].diff()
    gain = close_delta.clip(lower=0)
    loss = (-close_delta).clip(lower=0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs = avg_gain / avg_loss
    result["rsi_14"] = 100.0 - (100.0 / (1.0 + rs))

    rolling_mean_20 = result["close"].rolling(20).mean()
    rolling_std_price_20 = result["close"].rolling(20).std()
    upper_band = rolling_mean_20 + 2.0 * rolling_std_price_20
    lower_band = rolling_mean_20 - 2.0 * rolling_std_price_20
    result["bb_pctb"] = (result["close"] - lower_band) / (upper_band - lower_band)
    result["rolling_std_20"] = result["close"].pct_change().rolling(20).std()

    obv_direction = np.sign(result["close"].diff())
    obv_direction.iloc[0] = 0
    obv = (obv_direction * result["volume"]).cumsum()
    result["obv_roc"] = obv.pct_change(5)

    result[FEATURE_COLS] = result[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    return result.dropna(subset=FEATURE_COLS).reset_index(drop=True)


In [ ]:
def prepare_ticker_frame(ticker: str, max_raw_rows: int | None = None) -> pd.DataFrame:
    one_minute = load_one_minute_data(ticker, max_rows=max_raw_rows)
    regular_hours = filter_regular_hours(one_minute)
    five_minute = resample_to_five_minutes(regular_hours)
    feature_frame = add_technical_indicators(five_minute)
    if feature_frame.empty:
        raise ValueError(
            f"Ticker {ticker} has no usable rows after 5-minute resampling and feature cleanup"
        )
    feature_frame[TICKER_COL] = ticker
    return feature_frame


def apply_no_trade_band_labels(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    return make_no_trade_band_labels(
        df,
        price_col=PRICE_COL,
        k=LABEL_HORIZON_K,
        threshold_bps=THRESHOLD_BPS,
        timestamp_col=TIMESTAMP_COL,
    )


In [ ]:
def _make_window_dataset(frame: pd.DataFrame) -> WindowedClassificationDataset:
    return WindowedClassificationDataset(
        df=frame,
        feature_cols=FEATURE_COLS,
        label_col=LABEL_COL,
        ticker_col=TICKER_COL,
        timestamp_col=TIMESTAMP_COL,
        window_size=WINDOW_SIZE,
        label_horizon_k=LABEL_HORIZON_K,
        stride=1,
    )


def prepare_model_data(
    tickers: list[str],
    max_raw_rows_per_ticker: int | None = None,
):
    if not tickers:
        raise ValueError("tickers must be non-empty")

    data_config = DataConfig(
        tickers=tickers,
        data_dir=str(RAW_DATA_DIR),
        timestamp_col=TIMESTAMP_COL,
        price_col=PRICE_COL,
        label_mode="no_trade_band",
        threshold_bps=float(THRESHOLD_BPS),
        feature_cols=FEATURE_COLS,
        train_ratio=0.70,
        val_ratio=0.15,
        timezone_policy="naive",
    )

    labeled_frames = {}
    label_diagnostics = {}
    split_frames = {"train": {}, "val": {}, "test": {}}

    for ticker in tickers:
        feature_frame = prepare_ticker_frame(ticker, max_raw_rows=max_raw_rows_per_ticker)
        labeled_frame, diagnostics = apply_no_trade_band_labels(feature_frame)
        labeled_frames[ticker] = labeled_frame
        label_diagnostics[ticker] = diagnostics

        train_frame, val_frame, test_frame = make_time_splits(
            labeled_frame,
            train_ratio=data_config.train_ratio,
            val_ratio=data_config.val_ratio,
            timestamp_col=TIMESTAMP_COL,
            timezone_policy=data_config.timezone_policy,
        )
        split_frames["train"][ticker] = trim_labels_at_split_boundary(
            train_frame,
            label_horizon_k=LABEL_HORIZON_K,
            timestamp_col=TIMESTAMP_COL,
            ticker_col=TICKER_COL,
            timezone_policy=data_config.timezone_policy,
        )
        split_frames["val"][ticker] = trim_labels_at_split_boundary(
            val_frame,
            label_horizon_k=LABEL_HORIZON_K,
            timestamp_col=TIMESTAMP_COL,
            ticker_col=TICKER_COL,
            timezone_policy=data_config.timezone_policy,
        )
        split_frames["test"][ticker] = trim_labels_at_split_boundary(
            test_frame,
            label_horizon_k=LABEL_HORIZON_K,
            timestamp_col=TIMESTAMP_COL,
            ticker_col=TICKER_COL,
            timezone_policy=data_config.timezone_policy,
        )

    pooled_train_frame = pd.concat(
        [split_frames["train"][ticker] for ticker in tickers],
        ignore_index=True,
    )
    scaler = fit_scaler_on_train(pooled_train_frame, FEATURE_COLS, scaler_type=SCALER_TYPE)

    transformed = {"train": {}, "val": {}, "test": {}}
    datasets = {"train": {}, "val": {}, "test": {}}
    labels = {"train": {}, "val": {}, "test": {}}
    for split_name in ["train", "val", "test"]:
        for ticker in tickers:
            transformed[split_name][ticker] = transform_split(
                split_frames[split_name][ticker],
                scaler,
                FEATURE_COLS,
            )
            datasets[split_name][ticker] = _make_window_dataset(transformed[split_name][ticker])
            labels[split_name][ticker] = collect_dataset_labels(datasets[split_name][ticker])

    return {
        "data_config": data_config,
        "labeled_frames": labeled_frames,
        "label_diagnostics": label_diagnostics,
        "split_frames": split_frames,
        "transformed": transformed,
        "datasets": datasets,
        "labels": labels,
        "scaler": scaler,
    }


In [ ]:
from torch.utils.data import DataLoader


def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )


def collect_dataset_labels(dataset) -> np.ndarray:
    labels = [int(dataset[index][1].item()) for index in range(len(dataset))]
    return np.asarray(labels, dtype=np.int64)


def class_distribution(labels: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=np.int64)
    count = int(labels.size)
    up = int((labels == 1).sum())
    down = int((labels == 0).sum())
    return {
        "n": count,
        "up": up,
        "down": down,
        "up_pct": float(up / count) if count else np.nan,
        "down_pct": float(down / count) if count else np.nan,
    }


In [ ]:
def build_model(model_name: str):
    registry_by_name = {item["model_name"]: item for item in MODEL_REGISTRY}
    if model_name not in registry_by_name:
        raise ValueError(f"Unknown model_name {model_name!r}; expected one of {sorted(registry_by_name)}")
    registry_item = registry_by_name[model_name]
    return registry_item["constructor"](**registry_item["config"])


In [ ]:
def summarize_dummy_baselines(y_train: np.ndarray, y_target: np.ndarray, seed: int) -> dict:
    stratified_scores = []
    for offset in range(10):
        metrics = dummy_baseline_metrics(
            y_train=y_train,
            y_test=y_target,
            strategy="stratified",
            random_state=seed + offset,
        )
        stratified_scores.append(metrics["macro_f1"])

    prior_metrics = dummy_baseline_metrics(
        y_train=y_train,
        y_test=y_target,
        strategy="prior",
        random_state=seed,
    )
    always_up_metrics = always_predict_baseline_metrics(y_target, constant_label=1)
    always_down_metrics = always_predict_baseline_metrics(y_target, constant_label=0)
    return {
        "dummy_stratified_macro_f1_mean": float(np.mean(stratified_scores)),
        "dummy_stratified_macro_f1_std": float(np.std(stratified_scores)),
        "dummy_prior_macro_f1": float(prior_metrics["macro_f1"]),
        "always_up_macro_f1": float(always_up_metrics["macro_f1"]),
        "always_down_macro_f1": float(always_down_metrics["macro_f1"]),
    }


def make_result_row(
    *,
    model_name: str,
    ticker: str,
    seed: int,
    split: str,
    n_train_windows: int,
    n_val_windows: int,
    n_test_windows: int,
    label_retained_pct: float | None,
    model_metrics: dict,
    baseline_metrics: dict,
    confusion_matrix,
) -> dict:
    model_macro_f1 = model_metrics.get("macro_f1")
    dummy_macro_f1 = baseline_metrics.get("dummy_stratified_macro_f1_mean")
    delta_macro_f1 = None
    if model_macro_f1 is not None and dummy_macro_f1 is not None:
        delta_macro_f1 = float(model_macro_f1) - float(dummy_macro_f1)

    row = {
        "model_name": model_name,
        "ticker": ticker,
        "seed": seed,
        "window_size": WINDOW_SIZE,
        "label_horizon_k": LABEL_HORIZON_K,
        "threshold_bps": THRESHOLD_BPS,
        "split": split,
        "n_train_windows": n_train_windows,
        "n_val_windows": n_val_windows,
        "n_test_windows": n_test_windows,
        "label_retained_pct": label_retained_pct,
        "model_macro_f1": model_macro_f1,
        "model_balanced_accuracy": model_metrics.get("balanced_accuracy"),
        "dummy_stratified_macro_f1_mean": dummy_macro_f1,
        "dummy_stratified_macro_f1_std": baseline_metrics.get("dummy_stratified_macro_f1_std"),
        "dummy_prior_macro_f1": baseline_metrics.get("dummy_prior_macro_f1"),
        "always_up_macro_f1": baseline_metrics.get("always_up_macro_f1"),
        "always_down_macro_f1": baseline_metrics.get("always_down_macro_f1"),
        "delta_macro_f1_vs_dummy": delta_macro_f1,
        "confusion_matrix": confusion_matrix,
    }
    missing_columns = [column for column in NOTEBOOK03_RESULT_COLUMNS if column not in row]
    if missing_columns:
        raise ValueError(f"Result row missing required columns: {missing_columns}")
    return {column: row[column] for column in NOTEBOOK03_RESULT_COLUMNS}


In [ ]:
NOTEBOOK03_OPTIONAL_DIAGNOSTIC_COLUMNS = [
    "candidate_id",
    "candidate_name",
    "train_up_pct",
    "train_down_pct",
    "val_up_pct",
    "val_down_pct",
    "test_up_pct",
    "test_down_pct",
    "label_n_total",
    "label_n_retained",
    "label_n_up",
    "label_n_down",
    "label_n_neutral",
    "label_n_cross_day",
    "label_n_tail",
    "model_precision_macro",
    "model_recall_macro",
    "best_epoch",
    "best_val_macro_f1",
    "training_time_seconds",
    "suspicious_status",
]


In [ ]:
optional_diagnostic_schema_df = pd.DataFrame(columns=NOTEBOOK03_OPTIONAL_DIAGNOSTIC_COLUMNS)
optional_diagnostic_schema_df.columns


In [ ]:
def require_training_enabled() -> None:
    if not RUN_TRAINING:
        raise RuntimeError(
            "RUN_TRAINING is False. Notebook 03 is currently skeleton/smoke-only. "
            "Enable RUN_TRAINING only in a later explicit full-run step."
        )


def run_model_comparison():
    require_training_enabled()
    raise NotImplementedError(
        "Executable model comparison is staged but not enabled in P1B.16d. "
        "A later explicit full-run step must enable RUN_TRAINING and define execution policy."
    )


In [ ]:
import torch

smoke_x = torch.randn(2, WINDOW_SIZE, INPUT_SIZE)
smoke_rows = []
for registry_item in MODEL_REGISTRY:
    model = registry_item["constructor"](**registry_item["config"])
    model.eval()
    with torch.no_grad():
        logits = model(smoke_x)
    output_shape = tuple(logits.shape)
    assert output_shape == (2, NUM_CLASSES), (
        f"{registry_item['model_name']} returned {output_shape}, "
        f"expected {(2, NUM_CLASSES)}"
    )
    smoke_rows.append(
        {
            "model_name": registry_item["model_name"],
            "output_shape": output_shape,
        }
    )

smoke_shapes_df = pd.DataFrame(smoke_rows)
smoke_shapes_df

## Final Note

This notebook is intentionally not executed as a full run. The next step should be review plus atomic commit of the skeleton. Full Colab training requires a separate explicit prompt.